In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# =========================
# 2. Load Data
# =========================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

# =========================
# 3. Separate target
# =========================
target = 'Irrigation_Need'

X = train.drop(columns=[target, 'id'])
y = train[target]

X_test = test.drop(columns=['id'])
test_ids = test['id']

# =========================
# 4. Encode target ONLY
# =========================
target_le = LabelEncoder()
y = target_le.fit_transform(y)

# =========================
# 5. Identify categorical columns
# =========================
cat_cols = X.select_dtypes(include='object').columns.tolist()

# =========================
# 6. Train-validation split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# =========================
# 7. CatBoost Model
# =========================
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    class_weights=[1, 1, 5],   # boost "High"
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

# =========================
# 8. Train
# =========================
model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols,
    use_best_model=True
)

# =========================
# 9. Predict
# =========================
preds = model.predict(X_test)

# Convert back to original labels
preds = target_le.inverse_transform(preds.astype(int).flatten())

# =========================
# 10. Submission
# =========================
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': preds
})

submission.to_csv('submission.csv', index=False)

print("✅ CatBoost submission ready!")

0:	learn: 0.9741189	test: 0.9754996	best: 0.9754996 (0)	total: 1.81s	remaining: 30m 10s
100:	learn: 0.9783576	test: 0.9795215	best: 0.9795215 (100)	total: 2m 35s	remaining: 23m 6s
200:	learn: 0.9787004	test: 0.9797958	best: 0.9798052 (199)	total: 4m 54s	remaining: 19m 31s
300:	learn: 0.9790479	test: 0.9800637	best: 0.9800637 (300)	total: 7m 14s	remaining: 16m 48s
400:	learn: 0.9794411	test: 0.9804135	best: 0.9804135 (389)	total: 9m 45s	remaining: 14m 34s
500:	learn: 0.9796854	test: 0.9806121	best: 0.9806499 (491)	total: 12m 14s	remaining: 12m 11s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9806499401
bestIteration = 491

Shrink model to first 492 iterations.
✅ CatBoost submission ready!
